# Data cleaning — Financial KPI Dashboard

Loads `data/raw_data.csv`, fixes data-quality issues, and writes `data/cleaned_data.csv` for EDA and KPI analysis.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if (Path.cwd().name == "notebooks") else Path.cwd()
RAW = ROOT / "data" / "raw_data.csv"
OUT = ROOT / "data" / "cleaned_data.csv"

df = pd.read_csv(RAW, parse_dates=["Date"])
print("Shape (raw):", df.shape)
df.head()

## Duplicates
Remove exact duplicate rows (same Date, Region, Category, Revenue, Profit, Quantity).

In [ ]:
dup_before = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print("Duplicates removed:", dup_before)

## Missing values
- **Quantity**: if missing, impute with the median quantity for that Region × Category.
- **Revenue**: if missing, backfill from `Quantity ×` typical unit revenue for that segment (median Revenue/Quantity).
- **Profit**: if missing, estimate from median profit margin for that Region × Category applied to Revenue.

In [ ]:
seg_med_qty = df.groupby(["Region", "Category"])["Quantity"].transform("median")
df["Quantity"] = df["Quantity"].fillna(seg_med_qty)
df["Quantity"] = df["Quantity"].fillna(df["Quantity"].median())

med_rev = df.groupby(["Region", "Category"])["Revenue"].transform("median")
df["Revenue"] = df["Revenue"].fillna(med_rev)
df["Revenue"] = df["Revenue"].fillna(df["Revenue"].median())

df["_pm"] = df["Profit"] / df["Revenue"].replace(0, np.nan)
med_margin = df.groupby(["Region", "Category"])["_pm"].transform("median")
df["Profit"] = df["Profit"].fillna(df["Revenue"] * med_margin)
df["Profit"] = df["Profit"].fillna(df["Revenue"] * df["_pm"].median())
df.drop(columns=["_pm"], inplace=True)

df["Profit_Margin_Pct"] = np.where(
    df["Revenue"] > 0, (df["Profit"] / df["Revenue"]) * 100.0, np.nan
)

print("Remaining nulls:", df.isna().sum().sum())
df.describe(include="all").T

## Validation
Drop non-positive revenue rows (if any) and save.

In [ ]:
before = len(df)
df = df[df["Revenue"] > 0].copy()
print("Rows dropped (non-positive revenue):", before - len(df))

OUT.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT, index=False)
print("Wrote:", OUT, "— rows:", len(df))